# Library Imports

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from routingpy import OSRM
from routingpy.exceptions import RouterApiError
from itertools import product
from math import sqrt

# 2. SIM Model 

This section loads the Bulgarian census data and extended distance matrix, defines an updated SIM function that computes flows between settlements (and optionally returns the full flows matrix),and then evaluates the model over a grid of parameters.

## 2.1 Load Data and Setup Environment

In [ ]:
# Load Bulgarian Census Data
excel_path_bg = r"C:\Users\pietg\OneDrive - University of Glasgow\PhD Research\Phd_Data\PopulationGrid_TR_BG\Scripts\bg_2021_census\bg_census_merged_data_pivot.xlsx"
data_bg = pd.read_excel(excel_path_bg, sheet_name='data_bg_updated2')

# Define output path for saving results
output_path_bg = r"D:\PhD\SIM_Model_BG_unconstrained_iterative"
os.makedirs(output_path_bg, exist_ok=True)

In [ ]:
# Load OSRM Data matrix

# Load the OSRM NPZ file (make sure the file name/path is correct)
osrm_data = np.load('calculated_distance_matrices/distance_matrix_bg_3.npz', allow_pickle=True)
osrm_settlement_ids = np.array(osrm_data['settlement_ids'], dtype=str)  
# Convert distances from meters to km if needed:
osrm_distance_matrix_full = osrm_data['distance_matrix'] / 1000.0  


In [ ]:
# Define population columns for each census year
population_years_bg = ['y31_12_1934', 
                         'y31_12_1946', 'y01_12_1956', 'y01_12_1965', 
                         'y02_12_1975', 'y04_12_1985', 'y04_12_1992']

# Additional columns for reference output (if needed)
extra_columns_bg = ['village', 'municipality', 'district', 'ekatte_id', 'region', 'r_mean', 'r_median']

In [ ]:
# We use use the settlement data as is. No external point (for international migration) is appended.
updated_data_bg = data_bg.copy()

In [ ]:
# This script is needed to make sure that the settlement ids in the updated_data_bg are in the same order as the OSRM data. 
# This was an issue after an update to the excel data in which the order of the settlements changed. 
# Succesful ordering can be checked with the next code cell.

# Ensure the 'ekatte_id' column in updated_data_bg is a string
updated_data_bg['ekatte_id'] = updated_data_bg['ekatte_id'].astype(str)

# Filter updated_data_bg to include only settlements that exist in the OSRM data
updated_data_bg_sub = updated_data_bg[updated_data_bg['ekatte_id'].isin(osrm_settlement_ids)].copy()

# Reorder updated_data_bg_sub so its order matches that of osrm_settlement_ids.
# We'll loop through the osrm_settlement_ids and pick matching rows from updated_data_bg_sub.
ordered_rows = []
for uid in osrm_settlement_ids:
    # Only add rows for which a match exists
    if uid in updated_data_bg_sub['ekatte_id'].values:
        row = updated_data_bg_sub[updated_data_bg_sub['ekatte_id'] == uid]
        ordered_rows.append(row)
        
# Combine the ordered rows into a single DataFrame
updated_data_bg_ordered = pd.concat(ordered_rows, ignore_index=True)

# Reorder the Distance Matrix to Match updated_data_bg_ordered ---

# For each ekatte_id in updated_data_bg_ordered, find its index in osrm_settlement_ids.
indices = [np.where(osrm_settlement_ids == uid)[0][0] for uid in updated_data_bg_ordered['ekatte_id']]

# Extract the submatrix from the full OSRM distance matrix that corresponds to the ordered settlements.
distance_matrix_bg_ordered = osrm_distance_matrix_full[np.ix_(indices, indices)]

print("Reordered updated_data_bg shape:", updated_data_bg_ordered.shape)
print("Reordered distance_matrix shape:", distance_matrix_bg_ordered.shape)

In [ ]:
# Optional: Verify a specific O-D pair (should be 593.81 km for osrm based distance matrix):
id_origin = '68134'
id_destination = '39493'

# Find indices in the ordered DataFrame:
try:
    idx_origin = np.where(updated_data_bg_ordered['ekatte_id'] == id_origin)[0][0]
    idx_destination = np.where(updated_data_bg_ordered['ekatte_id'] == id_destination)[0][0]
    diag_distance = distance_matrix_bg_ordered[idx_origin, idx_destination]
    print(f"Distance for ekatte_id {id_origin} -> {id_destination} in reordered matrix: {diag_distance:.2f} km")
except IndexError:
    print("One or both ekatte_ids not found in the ordered DataFrame.")

## 2.2 calculation of flows: Compute Population Predictions for settlements based on Year Pair

In [ ]:
# --- Prepare the evaluation DataFrame using the ordered settlement data ---
evaluation_df = updated_data_bg_ordered.copy().reset_index(drop=True)
# rename lat and lon columns
evaluation_df["longitude"] = evaluation_df["x"]
evaluation_df["latitude"] = evaluation_df["y"]

In [ ]:
# Build the list of model parameter configurations.
param_list = []
for pop_scaling in np.arange(1.2, 2, 0.1):
    for dist_decay in np.arange(0.5, 3.5, 0.5):
        for model_type in ['gamma', 'exponential']:
            pop_scaling_r = round(pop_scaling, 1)
            dist_decay_r = round(dist_decay, 1)
            param_list.append((pop_scaling_r, dist_decay_r, model_type))

In [ ]:
# Prepare a list to collect prediction DataFrames
prediction_list = []

# Loop over each parameter configuration
for (pop_scaling, dist_decay, model_type) in param_list:
    print(f"Processing parameters: population_scaling={pop_scaling}, distance_decay={dist_decay}, model_type={model_type}")
    
    # Loop over each consecutive census interval
    for i in range(1, len(population_years_bg)):
        year_start = population_years_bg[i-1]
        year_end = population_years_bg[i]
        
        # Filter out settlements with 0 in either the year_start or year_end column
        mask = (evaluation_df[year_start].fillna(0) != 0) & (evaluation_df[year_end].fillna(0) != 0)
        filtered_df = evaluation_df[mask].copy()
        
        # Retrieve base and target populations (as float32 arrays, filling missing values with 0)
        pop_start = filtered_df[year_start].fillna(0).to_numpy(dtype=np.float32)
        pop_end   = filtered_df[year_end].fillna(0).to_numpy(dtype=np.float32)
        
        # Filter the distance matrix accordingly
        distance_matrix_filtered = distance_matrix_bg_ordered[mask, :][:, mask]
        
        # Compute the attractiveness matrix using the SIM formula.
        # Note: We use pop_end (observed target population) in the attractiveness.
        with np.errstate(divide='ignore', invalid='ignore'):
            if model_type == 'gamma':
                attractiveness = pop_start[:, None] * (pop_end ** pop_scaling) / (distance_matrix_filtered ** dist_decay)
            elif model_type == 'exponential':
                attractiveness = pop_start[:, None] * (pop_end ** pop_scaling) * np.exp(-dist_decay * distance_matrix_filtered)
            else:
                raise ValueError("Unknown model_type. Choose 'gamma' or 'exponential'.")
            attractiveness[~np.isfinite(attractiveness)] = 0
        
        # Remove self-flows by zeroing the diagonal.
        np.fill_diagonal(attractiveness, 0)
        
        # For each settlement, compute its total attractiveness (i.e., row sum)
        row_sums = np.sum(attractiveness, axis=1)
        total_attractiveness = np.sum(row_sums)
        
        # Compute total migration difference: the difference between total target and total base population.
        total_migration_diff = np.sum(pop_end) - np.sum(pop_start)
        
        # Compute a scaling factor to distribute the total migration difference.
        k = total_migration_diff / total_attractiveness if total_attractiveness != 0 else 0
        
        # Compute net migration for each settlement as: (row_sum * k)
        net_migration = row_sums * k
        
        # Predicted population is then base population plus the allocated net migration.
        predicted = pop_start + net_migration
        
        # Compute aggregate sums and fraction-based metrics for evaluation.
        sum_observed = np.sum(pop_end)
        sum_predicted = np.sum(predicted)
        
        fraction_observed = np.divide(
            pop_end,
            sum_observed,
            out=np.zeros_like(pop_end),
            where=(sum_observed != 0)
        )
        fraction_predicted = np.divide(
            predicted,
            sum_predicted,
            out=np.zeros_like(predicted),
            where=(sum_predicted != 0)
        )
        norm_residual = fraction_predicted - fraction_observed
        
        # Build a tidy DataFrame for this interval and parameter configuration.
        df_pred = filtered_df[['ekatte_id', 'longitude', 'latitude']].copy()
        df_pred["year_start"] = year_start
        df_pred["year_end"] = year_end
        df_pred["observed"] = pop_end
        df_pred["predicted"] = predicted
        df_pred["sum_observed"] = sum_observed
        df_pred["sum_predicted"] = sum_predicted
        df_pred["fraction_observed"] = fraction_observed
        df_pred["fraction_predicted"] = fraction_predicted
        df_pred["norm_residual"] = norm_residual
        
        # Store the model parameters for reference.
        df_pred["population_scaling"] = pop_scaling
        df_pred["distance_decay"] = dist_decay
        df_pred["model_type"] = model_type
        
        # Append the DataFrame to the list of predictions.
        prediction_list.append(df_pred)

# Concatenate all the predictions across intervals and parameter configurations.
predictions_df = pd.concat(prediction_list, ignore_index=True)

# Ensure that ekatte_id is stored as a string.
predictions_df["ekatte_id"] = predictions_df["ekatte_id"].astype(str)

In [ ]:
# Save the predictions DataFrame as a Parquet file using pyarrow and snappy compression.
predictions_df.to_parquet(
    os.path.join(output_path_bg, "predictions_by_interval_1602025.parquet"),
    engine="pyarrow",
    compression="snappy"
)

print("Saved predictions with fraction-of-total approach, storing sums and fraction columns.")


In [ ]:
# after saving the predictions were evaluated using DBeaver as a SQL query and evaluation criteria were calculated there as well.
# The SQL query was as follows:

'''
-- new script with SRMSE value:
COPY (
WITH sub AS (
  SELECT
    year_start,
    year_end,
    population_scaling,
    distance_decay,
    model_type,
    observed,
    predicted,
    -- Compute the mean of observed for each group (all values in the group are identical)
    AVG(observed) OVER (
      PARTITION BY
        year_start,
        year_end,
        population_scaling,
        distance_decay,
        model_type
    ) AS observed_mean
  FROM "D:\PhD\SIM_Model_BG_unconstrained_iterative\predictions_by_interval_1602025.parquet"
)
SELECT
  year_start,
  year_end,
  population_scaling,
  distance_decay,
  model_type,
  -- Root Mean Squared Error (RMSE)
  SQRT(AVG(POWER(observed - predicted, 2))) AS rmse,
  -- Mean Absolute Error (MAE)
  AVG(ABS(observed - predicted)) AS mae,
  -- Coefficient of Determination (R^2)
  1 - (
    SUM(POWER(observed - predicted, 2)) /
    NULLIF(SUM(POWER(observed - observed_mean, 2)), 0)
  ) AS r2,
  -- Relative Root Mean Squared Error (RRMSE)
  CASE
    WHEN MIN(observed_mean) = 0 THEN NULL
    ELSE SQRT(AVG(POWER(observed - predicted, 2))) / MIN(observed_mean)
  END AS rrmse,
  -- Mean Absolute Percentage Error (MAPE)
  AVG(ABS((observed - predicted) / NULLIF(observed, 0))) * 100 AS mape,
  -- Standardized Root Mean Squared Error (SRMSE)
  SQRT(SUM(POWER(observed - predicted, 2))) / (SQRT(COUNT(*)) * MIN(observed_mean)) AS srmse
FROM sub
GROUP BY
  year_start,
  year_end,
  population_scaling,
  distance_decay,
  model_type
ORDER BY
  CAST(RIGHT(year_start, 4) AS INT),
  population_scaling,
  distance_decay
)
TO 'D:\PhD\SIM_Model_BG_unconstrained_iterative\predictions_by_interval_evaluation_16042025_v2.csv' (HEADER, DELIMITER ',');

'''


In [ ]:
# if best_params is not loaded
interval_df = pd.read_csv(r"D:\PhD\SIM_Model_BG_unconstrained_iterative\predictions_by_interval_evaluation_16042025.csv")
interval_df['population_scaling'] = interval_df['population_scaling'].astype(float)
interval_df['distance_decay'] = interval_df['distance_decay'].astype(float)

# here we choose which model to use for the best parameters
best_params = interval_df.loc[interval_df.groupby(['year_start', 'year_end'])['srmse'].idxmin()].reset_index(drop=True)
best_params = best_params[['year_start', 'year_end', 'population_scaling', 'distance_decay', 'model_type']]

In [ ]:
# Now we will compute the OD flows for each year pair for each settlement and save them as a Parquet file. 
# This is without net migration and as full list, because the full list of o-d flows is needed to evaluate the inflows and outflows with the Philipov 1976 flows

import numpy as np
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
import gc

def print_mem_info(arr, name):
    if isinstance(arr, pd.DataFrame):
        mem = arr.memory_usage(deep=True).sum() / 1e6
        shape = arr.shape
    else:
        mem = arr.nbytes / 1e6
        shape = arr.shape
    print(f"{name} shape: {shape}, memory usage: {mem:.2f} MB")

# -----------------------------------------------------------------
# Assumptions:
# - best_params: DataFrame with columns ['year_start','year_end','population_scaling','distance_decay','model_type']
# - updated_data_bg_ordered: ordered settlement DataFrame with at least 'ekatte_id', 'x' or 'longitude', 'y' or 'latitude',
#   and extra_columns_bg.
# - distance_matrix_bg_ordered: OSRM distance matrix (in km) aligned with updated_data_bg_ordered.
# - extra_columns_bg: e.g., ['village', 'municipality', 'district', 'region']
# - output_path_bg: output directory for saving files.
# -----------------------------------------------------------------

# Define output directory for OD flows.
od_out_dir = os.path.join(output_path_bg, "OD_Flows_by_YearPair_srmse")
os.makedirs(od_out_dir, exist_ok=True)

# Use the ordered settlement DataFrame.
evaluation_df = updated_data_bg_ordered.copy().reset_index(drop=True)
if "x" in evaluation_df.columns and "y" in evaluation_df.columns:
    evaluation_df["longitude"] = evaluation_df["x"]
    evaluation_df["latitude"] = evaluation_df["y"]

print_mem_info(evaluation_df, "Evaluation DF")

# Create a single output file for all year pairs.
out_filename = os.path.join(od_out_dir, "OD_Flows_all_years.geoparquet")
writer = None  # Will be initialized with the first chunk

# Define a chunk size (number of OD pairs per write). Adjust this as needed.
chunk_size = int(1e6)

# Loop over each best parameter configuration (each representing a year pair).
for idx, row in best_params.iterrows():
    yr_start = row['year_start']
    yr_end = row['year_end']
    pop_scaling = float(row['population_scaling'])
    dist_decay = float(row['distance_decay'])
    model_type = row['model_type']
    
    print("\n" + "="*60)
    print(f"Processing OD flows for {yr_start} -> {yr_end} with parameters: scaling={pop_scaling}, decay={dist_decay}, model_type={model_type}")
    
    # --- Filter settlements based on non-zero populations in both years ---
    mask_settlements = (evaluation_df[yr_start].fillna(0) != 0) & (evaluation_df[yr_end].fillna(0) != 0)
    filtered_df = evaluation_df[mask_settlements].copy()
    print(f"Filtered settlements from {evaluation_df.shape[0]} to {filtered_df.shape[0]} based on non-zero populations for {yr_start} and {yr_end}")
    
    # Retrieve base and target populations.
    pop_start = filtered_df[yr_start].fillna(0).to_numpy(dtype=np.float32)
    pop_end   = filtered_df[yr_end].fillna(0).to_numpy(dtype=np.float32)
    N = pop_start.shape[0]
    print(f"Number of settlements after filtering: {N}")
    
    # Filter the distance matrix accordingly.
    distance_matrix_filtered = distance_matrix_bg_ordered[mask_settlements, :][:, mask_settlements]
    
    # Compute attractiveness matrix A using the chosen model.
    if model_type.lower() == 'gamma':
        A = pop_start[:, None] * (pop_end[None, :] ** pop_scaling) / (distance_matrix_filtered ** dist_decay)
    elif model_type.lower() == 'exponential':
        A = pop_start[:, None] * (pop_end[None, :] ** pop_scaling) * np.exp(-dist_decay * distance_matrix_filtered)
    else:
        raise ValueError("Unknown model_type. Choose 'gamma' or 'exponential'.")
    
    A[~np.isfinite(A)] = 0
    np.fill_diagonal(A, 0)
    print_mem_info(A, "Attractiveness Matrix A")
    
    A_total = A.sum()
    M_total = pop_end.sum() - pop_start.sum()
    k = M_total / A_total if A_total != 0 else 0
    print(f"Total A: {A_total:.2f}, Total migration diff: {M_total:.2f}, k: {k:.6f}")
    
    # Compute flows matrix F.
    F = k * A
    np.fill_diagonal(F, 0)
    print_mem_info(F, "Flows Matrix F")
    
    # --- Flatten the full matrix for all possible pairs (exclude self-flows) ---
    all_orig_idx = np.repeat(np.arange(N), N)
    all_dest_idx = np.tile(np.arange(N), N)
    mask = all_orig_idx != all_dest_idx
    all_orig_idx = all_orig_idx[mask]
    all_dest_idx = all_dest_idx[mask]
    num_pairs = len(all_orig_idx)
    print(f"Extracted {num_pairs} OD pairs (including both directions).")
    
    predicted_flow = F[all_orig_idx, all_dest_idx]
    distances = distance_matrix_filtered[all_orig_idx, all_dest_idx]
    
    # Process and write in chunks to avoid memory issues.
    for start in range(0, num_pairs, chunk_size):
        end = min(start + chunk_size, num_pairs)
        idx_slice = slice(start, end)
        
        batch_orig_idx = all_orig_idx[idx_slice]
        batch_dest_idx = all_dest_idx[idx_slice]
        
        # Build the output dictionary for the chunk.
        od_data = {
            'origin_ekatte': filtered_df.iloc[batch_orig_idx]['ekatte_id'].values,
            'destination_ekatte': filtered_df.iloc[batch_dest_idx]['ekatte_id'].values,
            'origin_longitude': (filtered_df.iloc[batch_orig_idx]['x'].values 
                                 if 'x' in filtered_df.columns 
                                 else filtered_df.iloc[batch_orig_idx]['longitude'].values),
            'origin_latitude': (filtered_df.iloc[batch_orig_idx]['y'].values 
                                if 'y' in filtered_df.columns 
                                else filtered_df.iloc[batch_orig_idx]['latitude'].values),
            'destination_longitude': (filtered_df.iloc[batch_dest_idx]['x'].values 
                                      if 'x' in filtered_df.columns 
                                      else filtered_df.iloc[batch_dest_idx]['longitude'].values),
            'destination_latitude': (filtered_df.iloc[batch_dest_idx]['y'].values 
                                     if 'y' in filtered_df.columns 
                                     else filtered_df.iloc[batch_dest_idx]['latitude'].values),
            'predicted_flow': predicted_flow[idx_slice],
            'distance_km': distances[idx_slice],
            'year_start': yr_start,
            'year_end': yr_end,
            'population_scaling': pop_scaling,
            'distance_decay': dist_decay,
            'model_type': model_type
        }
        
        # Add extra columns for origin and destination.
        for col in extra_columns_bg:
            od_data[f'origin_{col}'] = filtered_df.iloc[batch_orig_idx][col].values
            od_data[f'destination_{col}'] = filtered_df.iloc[batch_dest_idx][col].values
        
        od_df = pd.DataFrame(od_data)
        print_mem_info(od_df, f"OD DF chunk {start}-{end}")
        
        # Convert the DataFrame to a PyArrow Table.
        table = pa.Table.from_pandas(od_df, preserve_index=False)
        
        # Initialize the writer with the first chunk.
        if writer is None:
            writer = pq.ParquetWriter(out_filename, table.schema, compression='snappy')
        
        # Write the current chunk to the file.
        writer.write_table(table)
        print(f"Appended chunk {start} - {end} for {yr_start} -> {yr_end} to {out_filename}")
        
        del od_df, table
        gc.collect()
    
    # Clean up memory for this year pair.
    del A, F, predicted_flow, distances, all_orig_idx, all_dest_idx
    gc.collect()

# Close the writer once all year pairs are processed.
if writer is not None:
    writer.close()

print("Finished processing all year pairs.")


In [ ]:
import duckdb
import os

# -----------------------------------------------------------------
# Assumptions:
# - output_path_bg is defined.
# - The input geoparquet file "OD_Flows_all_years.geoparquet" exists in od_out_dir.
# - The input file contains at least:
#       origin_ekatte, destination_ekatte, predicted_flow,
#       year_start, year_end, population_scaling, distance_decay, model_type,
#       origin_longitude, origin_latitude, destination_longitude, destination_latitude,
#       distance_km,
#   plus any extra columns you want to carry over.
# -----------------------------------------------------------------

# Define file paths.
od_out_dir = os.path.join(output_path_bg, "OD_Flows_by_YearPair_srmse")
input_file = os.path.join(od_out_dir, "OD_Flows_all_years.geoparquet")
final_output_file = os.path.join(od_out_dir, "Aggregated_OD_Flows_with_all_extras_and_geom.parquet")

con = duckdb.connect()

# Install and load spatial extension if needed.
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

agg_query = f"""
COPY (
WITH base AS (
    SELECT *
    FROM read_parquet('{input_file}')
    WHERE origin_ekatte < destination_ekatte
)
SELECT
    base.*,  -- include all columns from table a (i.e. base)
    base.predicted_flow AS outflow,
    COALESCE(b.predicted_flow, 0) AS inflow,
    COALESCE(b.predicted_flow, 0) - base.predicted_flow AS net_migration,
    ST_MakeLine(
        ST_Point(base.origin_longitude, base.origin_latitude),
        ST_Point(base.destination_longitude, base.destination_latitude)
    ) AS geometry
FROM base
LEFT JOIN read_parquet('{input_file}') AS b
    ON base.year_start = b.year_start
    AND base.year_end = b.year_end
    AND base.population_scaling = b.population_scaling
    AND base.distance_decay = b.distance_decay
    AND base.model_type = b.model_type
    AND base.origin_ekatte = b.destination_ekatte
    AND base.destination_ekatte = b.origin_ekatte
) TO '{final_output_file}' (FORMAT 'parquet', COMPRESSION 'snappy');
"""

print("Aggregating OD flows with DuckDB (including all extra columns and geometry) ...")
con.execute(agg_query)
con.close()
print(f"Saved final aggregated flows with geometry and extra columns to {final_output_file}")


# Calculate the flows to compare with the o-d matrix data of 1976 D. Philipov

In [ ]:
import pandas as pd
import os

# Define the path to your OD flows GeoParquet file for the 1965-1976 period.
parquet_file = r"D:\PhD\SIM_Model_BG_unconstrained_iterative\OD_Flows_by_YearPair\OD_Flows_1965_1975.geoparquet"
# Load the parquet file (GeoParquet can be loaded as a normal DataFrame if geometry is not needed)
df = pd.read_parquet(parquet_file)

# Verify that the region columns exist. They should be named, e.g., 'origin_region' and 'destination_region'.
print("Unique origin regions:", df['origin_region'].unique())
print("Unique destination regions:", df['destination_region'].unique())

# Aggregate flows by region.
# You can aggregate the predicted flows (for example, using predicted_outflow and predicted_inflow)
# or the net_migration. Here, we'll sum the predicted outflow, inflow, and net_migration.
agg_df = df.groupby(['origin_region', 'destination_region'])[['predicted_outflow', 'predicted_inflow', 'net_migration']].sum().reset_index()

# Annualize flows by dividing by 10 (since the period spans roughly 10 years)
agg_df['predicted_outflow_annual'] = agg_df['predicted_outflow'] / 10
agg_df['predicted_inflow_annual'] = agg_df['predicted_inflow'] / 10
agg_df['net_migration_annual'] = agg_df['net_migration'] / 10

# Keep only the columns of interest.
final_df = agg_df[['origin_region', 'destination_region', 
                   'predicted_outflow_annual', 'predicted_inflow_annual', 'net_migration_annual']]

# Optionally, sort the table by origin and destination regions.
final_df = final_df.sort_values(by=['origin_region', 'destination_region'])

# Print the table.
print(final_df.head())


# aggregated table to a CSV or text file.
output_csv = os.path.join(os.path.dirname(parquet_file), "aggregated_region_flows_2_1965_1976.csv")
final_df.to_csv(output_csv, index=False, float_format="%.2f")
print(f"Aggregated flows saved to {output_csv}")